# M2 · Customers Cleaning (pandas)

**Goal:** turn the raw CRM export (50,000 rows) into one clean, trustworthy **golden record per customer**.

Every step refers to an `issue_id` from the M1 issue log (`01_profiling-pandas/output/issue_log.xlsx`). Values are only changed when the correct value can be proven; anything uncertain is **flagged, not guessed**.

**Outputs**
| File | Folder | Content |
|---|---|---|
| `customers_clean.xlsx` | `clean_data/` | sheets `customers` (golden records), `customer_devices`, `flagged` (records that need review, with the reason), `merge_map` (what happened to every raw row) |
| `customers_clean.csv`, `customer_devices.csv` | `clean_data/csv/` | the same clean data as UTF-8 CSV |
| `m2_fix_log.xlsx` | `output/` (this folder) | every fix and flag, linked to its M1 `issue_id` |

## 1. Setup

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW_DIR = Path("../../01_data_and_task/raw_data")
CLEAN_DIR = Path("../../03_clean_data_and_report/clean_data")
OUT_DIR = Path("output")  # detailed working outputs of this milestone (logs, charts)
OUT_DIR.mkdir(exist_ok=True)
ISSUE_LOG = Path("../01_profiling-pandas/output/issue_log.xlsx")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

READ_OPTS = dict(dtype=str, keep_default_na=False, encoding="utf-8")

raw = pd.read_csv(RAW_DIR / "crm_50000_customers_dirty_v3.csv", **READ_OPTS)
DATA_COLS = raw.columns.tolist()
# raw_row = line number in the CSV file (line 1 is the header), so every row can be traced back to the source
raw.insert(0, "raw_row", np.arange(2, len(raw) + 2))
print(f"raw customers: {len(raw):,} rows x {len(DATA_COLS)} columns")

issue_log = pd.read_excel(ISSUE_LOG, sheet_name="issue_log")
m2_issues = issue_log[issue_log["milestone"] == "M2"]
m2_issues[["issue_id", "column", "issue", "rows_affected"]]

raw customers: 50,000 rows x 14 columns


,issue_id,column,issue,rows_affected
0,ISS-001,phone_number,"phone numbers in 21 different formats, none in...",50000
1,ISS-002,country,US state combined with a non-US country (state...,49799
2,ISS-003,phone_number,phone extension mixed into the number,29966
3,ISS-004,phone_number,invalid US area code (starts with 0 or 1),10065
4,ISS-005,last_name,inconsistent letter case,6280
5,ISS-006,first_name,inconsistent letter case,6272
6,ISS-007,last_name,leading/trailing spaces,3134
7,ISS-008,first_name,leading/trailing spaces,3124
8,ISS-009,first_name,noise characters (digits or symbols) in name,3104
9,ISS-010,last_name,noise characters (digits or symbols) in name,3086


## 2. Fix log helpers

`log_fix` records what was done for each M1 issue: how many rows were **fixed** (value corrected) or **flagged** (kept as is, marked for review).

In [2]:
fix_log = []
flags = []


def iid(column, issue_start):
    """Find the M1 issue_id for this column whose description starts with issue_start."""
    hit = m2_issues[(m2_issues["column"] == column) & m2_issues["issue"].str.startswith(issue_start)]
    assert len(hit) == 1, f"expected one issue for {column!r} / {issue_start!r}, found {len(hit)}"
    return hit["issue_id"].iloc[0]


def log_fix(issue_id, action, fixed=0, flagged=0):
    row = m2_issues.set_index("issue_id").loc[issue_id]
    fix_log.append({
        "issue_id": issue_id, "column": row["column"], "issue": row["issue"],
        "rows_in_m1": int(row["rows_affected"]), "rows_fixed": int(fixed), "rows_flagged": int(flagged),
        "action_taken": action,
    })


def flag(df, mask, column, reason):
    """Store one review flag per affected customer."""
    hit = pd.DataFrame({
        "customer_id": df.loc[mask, "customer_id"],
        "column": column,
        "value": df.loc[mask, column].astype(str),
        "reason": reason,
    })
    flags.append(hit)
    return int(mask.sum())

## 3. Exact duplicate rows

Rows that are identical to an earlier row in every column are dropped (first copy kept). This is decided on the **raw** values, before any change.

In [3]:
exact_dup = raw.duplicated(subset=DATA_COLS)
merge_map = raw[["raw_row", "customer_id"]].copy()
merge_map["action"] = np.where(exact_dup, "dropped: exact duplicate of an earlier row", "")

df = raw.loc[~exact_dup].copy()
log_fix(iid("(all)", "exact duplicate row"), "Dropped, first occurrence kept", fixed=exact_dup.sum())
print(f"dropped {exact_dup.sum():,} exact duplicates -> {len(df):,} rows")

dropped 1,021 exact duplicates -> 48,979 rows


## 4. Trim spaces

In [4]:
for col in DATA_COLS:
    trimmed = df[col].str.strip()
    changed = df[col].ne(trimmed)
    if changed.any() and col in ("first_name", "last_name"):
        log_fix(iid(col, "leading/trailing spaces"), "Trimmed", fixed=changed.sum())
    df[col] = trimmed

## 5. Names

Three problems, fixed in this order:
1. **Noise characters** (digits, `*`, `!`, `@`) are removed.
2. **Doubled-letter typo**: the corruption doubles the first three letters (`KKeevvin`, `HHaannnah`, `JJeennnifer`). When the first three letter pairs are all identical, the original is rebuilt as letters 1, 3, 5 plus the rest (`Kevin`, `Hannah`, `Jennifer`). No genuine name starts with three doubled pairs, so this rule cannot damage a correct name.
3. **Letter case**: converted to title case (`mARIA` → `Maria`).

The result is checked against the email local part (`first.last###@...`) in section 12.

In [5]:
def fix_doubled(name):
    n = name.lower()
    if len(n) >= 6 and n[0] == n[1] and n[2] == n[3] and n[4] == n[5]:
        return name[0] + name[2] + name[4] + name[6:]
    return name


for col in ["first_name", "last_name"]:
    before = df[col].copy()

    no_noise = df[col].str.replace(r"[^A-Za-z '\-]", "", regex=True).str.strip()
    noise_fixed = no_noise.ne(before)

    undoubled = no_noise.astype(object).map(fix_doubled)
    doubled_fixed = undoubled.ne(no_noise)

    titled = undoubled.str.title()
    case_fixed = titled.ne(undoubled)

    df[col] = titled
    log_fix(iid(col, "noise characters"), "Removed digits and symbols", fixed=noise_fixed.sum())
    log_fix(iid(col, "possible doubled-letter typo"), "Rebuilt from letters 1, 3, 5 + rest", fixed=doubled_fixed.sum())
    log_fix(iid(col, "inconsistent letter case"), "Converted to title case", fixed=case_fixed.sum())
    print(f"{col}: noise {noise_fixed.sum():,} | doubled {doubled_fixed.sum():,} | case {case_fixed.sum():,}")

examples = raw.loc[df.index, ["first_name", "last_name"]].join(df[["first_name", "last_name"]], rsuffix="_clean")
examples[examples["first_name"].ne(examples["first_name_clean"])].head(10)

first_name: noise 3,104 | doubled 1,576 | case 4,696
last_name: noise 3,086 | doubled 1,581 | case 4,699


,first_name,last_name,first_name_clean,last_name_clean
0,mARIA,day,Maria,Day
1,KKeevvin,Cantu,Kevin,Cantu
9,JAY,JJoonnes,Jay,Jones
19,Tho8mas,Weeks,Thomas,Weeks
21,Caitlin,Hernandez,Caitlin,Hernandez
22,Jose,Kirk,Jose,Kirk
25,KEVIN,Ha0rt,Kevin,Hart
27,michael,sMITH,Michael,Smith
30,Dana,Smith,Dana,Smith
32,travis,Thomas,Travis,Thomas


## 6. Email

- Blank email becomes a real null (it cannot be recovered).
- `shared####@...` addresses are kept but marked with `email_is_shared`; they are generic inboxes and are never used to match customers.

In [6]:
df["email"] = df["email"].str.lower().replace("", pd.NA)
df["email_is_shared"] = df["email"].str.match(r"^shared\d+@", na=False)

log_fix(iid("email", "blank value"), "Set to null (no reliable source to fill)", flagged=df["email"].isna().sum())
log_fix(iid("email", "generic shared address"), "Kept; marked in column email_is_shared",
        flagged=df["email_is_shared"].sum())
assert df["email"].dropna().str.match(r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$").all()
print(f"null emails: {df['email'].isna().sum():,} | shared: {df['email_is_shared'].sum():,}")

null emails: 1,018 | shared: 1,516


## 7. Phone to E.164

All numbers are North American (country code 1). Steps:
1. Split off the extension (`x1234`) into `phone_ext`.
2. Keep digits only and remove the country prefix (`1` or `001`).
3. A valid NANP number has 10 digits, and both the area code (digit 1) and the exchange code (digit 4) start with 2 to 9.
4. Valid numbers become `+1XXXXXXXXXX`. Invalid ones stay null in `phone_e164` and are flagged; the original is kept in `phone_raw`.

In [7]:
phone = df["phone_number"]
parts = phone.str.split("x", n=1, expand=True)
digits = parts[0].str.replace(r"\D", "", regex=True)
digits = digits.where(~(digits.str.len().eq(13) & digits.str.startswith("001")), digits.str[3:])
digits = digits.where(~(digits.str.len().eq(11) & digits.str.startswith("1")), digits.str[1:])

valid = digits.str.match(r"^[2-9]\d{2}[2-9]\d{6}$")
df["phone_raw"] = phone
df["phone_e164"] = ("+1" + digits).where(valid)
df["phone_ext"] = parts[1].replace("", pd.NA) if parts.shape[1] > 1 else pd.NA

bad_area = ~digits.str.match(r"^[2-9]")
bad_exchange = digits.str.match(r"^[2-9]") & ~valid
n_area = flag(df, bad_area, "phone_raw", "invalid US area code (starts with 0 or 1); not converted to E.164")
n_exch = flag(df, bad_exchange, "phone_raw", "invalid US exchange code (digit 4 is 0 or 1); not converted to E.164")

log_fix(iid("phone_number", "phone numbers in"), "Converted valid numbers to E.164 (+1XXXXXXXXXX)", fixed=valid.sum())
log_fix(iid("phone_number", "phone extension"), "Moved extension to column phone_ext", fixed=df["phone_ext"].notna().sum())
log_fix(iid("phone_number", "invalid US area code"),
        "Not converted; original kept in phone_raw and flagged (area or exchange code invalid)",
        flagged=n_area + n_exch)
print(f"E.164: {valid.sum():,} | invalid area code: {n_area:,} | invalid exchange code: {n_exch:,}")
df[["phone_raw", "phone_e164", "phone_ext"]].head(8)

E.164: 31,291 | invalid area code: 9,873 | invalid exchange code: 7,815


,phone_raw,phone_e164,phone_ext
0,449.977.1729x282,+14499771729,282
1,501.347.4824x64633,+15013474824,64633
2,+1-268-822-6332x26814,+12688226332,26814
3,864-894-1347,+18648941347,NaN
4,489.732.5436,+14897325436,NaN
5,(522)382-8559,+15223828559,NaN
6,001-063-045-0948x4636,NaN,4636
7,755.281.0372,+17552810372,NaN


## 8. Dates

`dob` and `signup_date` are already ISO dates (M1 found none unparseable). They are converted to real dates. Customers younger than 13 at signup are kept but flagged: most online services do not allow this, so the birth date or signup date may be wrong.

In [8]:
df["dob"] = pd.to_datetime(df["dob"], format="%Y-%m-%d")
df["signup_date"] = pd.to_datetime(df["signup_date"], format="%Y-%m-%d")
age_at_signup = (df["signup_date"] - df["dob"]).dt.days / 365.25
young = age_at_signup < 13
n_young = flag(df, young, "signup_date", "customer younger than 13 at signup")
log_fix(iid("signup_date", "customer younger than 13"), "Kept and flagged for review", flagged=n_young)

## 9. Address and location

- All-caps addresses are converted to capitalised words (`37038 WILLIAMS RAMP SUITE 789` → `37038 Williams Ramp Suite 789`).
- `state` is always a US state, but `country` is a random country in 99.6% of rows. The source does not tell which field is right, so neither is changed: the column `state_country_conflict` marks these rows.

In [9]:
def capwords(text):
    return " ".join(w.capitalize() for w in text.split())


caps = df["address"].str.isupper()
df.loc[caps, "address"] = df.loc[caps, "address"].astype(object).map(capwords)
log_fix(iid("address", "address written in all caps"), "Converted to capitalised words", fixed=caps.sum())

df["state_country_conflict"] = ~df["country"].str.lower().isin(["united states", "united states of america"])
log_fix(iid("country", "US state combined"), "Not changed; marked in column state_country_conflict",
        flagged=df["state_country_conflict"].sum())
df.loc[caps, ["address"]].head()

,address
41,52798 Caleb Meadow Apt. 352
191,65226 Mary Drive
223,877 Omar Port Suite 748
238,992 Cervantes Fort Suite 026
297,42927 Mcintyre Lakes Suite 620


## 10. Golden record per customer_id

After the fixes above, rows that share a `customer_id` are compared again. If they are now identical, the differences were only formatting noise and they collapse into one golden record. If any field still conflicted, the first row would be kept and the customer flagged, so no value is silently chosen.

In [10]:
compare_cols = [c for c in df.columns if c not in ("raw_row",)]
same_id = df["customer_id"].duplicated(keep=False)
groups = df[same_id].groupby("customer_id")
conflicts = groups[compare_cols].nunique().gt(1)
conflict_ids = conflicts.index[conflicts.any(axis=1)]
print(f"customer_ids with more than one row: {groups.ngroups:,}")
print(f"still conflicting after normalisation: {len(conflict_ids):,}")
if len(conflict_ids):
    print(conflicts.loc[conflict_ids].sum())

dup_after = df["customer_id"].duplicated()
merge_map.loc[df.index[dup_after], "action"] = "merged: same customer_id, identical after normalisation"
customers = df.loc[~dup_after].copy()
flag(customers, customers["customer_id"].isin(conflict_ids), "customer_id",
     "same customer_id had conflicting values; first row kept")

log_fix(iid("customer_id", "same customer_id repeated"),
        f"Merged into one golden record; {len(conflict_ids)} conflicts left after normalisation",
        fixed=dup_after.sum(), flagged=len(conflict_ids))
merge_map.loc[merge_map["action"].eq(""), "action"] = "kept: golden record"
print(f"golden records: {len(customers):,}")
merge_map["action"].value_counts()

customer_ids with more than one row: 768
still conflicting after normalisation: 0
golden records: 48,200


action
kept: golden record                                        48200
dropped: exact duplicate of an earlier row                  1021
merged: same customer_id, identical after normalisation      779
Name: count, dtype: int64

## 11. Devices

`device_id(s)` packs several IDs into one cell (`id1;id2;id3`). They are moved to a separate `customer_devices` table with one row per customer-device pair, and the customer table keeps only `device_count`.

In [11]:
customer_devices = (
    customers[["customer_id", "device_id(s)"]]
    .assign(device_id=lambda d: d["device_id(s)"].str.split(";"))
    .explode("device_id")[["customer_id", "device_id"]]
    .assign(device_id=lambda d: d["device_id"].str.strip())
    .drop_duplicates()
    .reset_index(drop=True)
)
customers["device_count"] = customers["customer_id"].map(customer_devices["customer_id"].value_counts())
multi = customers["device_id(s)"].str.contains(";")
log_fix(iid("device_id(s)", "several device IDs"), "Split into table customer_devices", fixed=multi.sum())
print(f"customer-device pairs: {len(customer_devices):,}")
customers["device_count"].value_counts().sort_index()

customer-device pairs: 51,854


device_count
1    46999
2      234
3      229
4      239
5      250
6      249
Name: count, dtype: int64

## 12. Cross-checks: possible duplicates and email-name mismatch

These are **not** merged automatically, because two different customers can share a name and birth date, and an email can be reused. They are listed for review:
- the same personal (non-shared) email on different `customer_id`s
- the same first name, last name and date of birth on different `customer_id`s
- an email whose `first.last` part does not match the customer's name (the email may belong to someone else)

In [12]:
personal = customers["email"].notna() & ~customers["email_is_shared"]
same_email = personal & customers.groupby("email")["customer_id"].transform("nunique").gt(1)
n_email = flag(customers, same_email, "email", "same personal email on different customer_id (possible duplicate)")
log_fix(iid("email", "same personal email"), "Not merged; flagged as possible duplicate", flagged=n_email)

person = customers["first_name"].str.lower() + "|" + customers["last_name"].str.lower() + "|" + customers["dob"].astype(str)
same_person = person.duplicated(keep=False)
n_person = flag(customers, same_person, "customer_id", "same name and date of birth on different customer_id (possible duplicate)")
print(f"possible duplicates: same email {n_email:,} | same name + dob {n_person:,}")

local = customers["email"].str.extract(r"^([a-z]+)\.([a-z]+)\d*@")
has_local = local[0].notna()
first_ok = customers["first_name"].str.lower().eq(local[0])
last_ok = customers["last_name"].str.lower().eq(local[1])
print(f"emails in first.last format: {has_local.sum():,}")
print(f"  first name matches email: {first_ok[has_local].mean():.2%}")
print(f"  last name matches email:  {last_ok[has_local].mean():.2%}")
partial = has_local & (first_ok ^ last_ok)
print(f"  only one of the two matches: {partial.sum():,}")

# Reviewed: in every partial case the raw name already equals the cleaned name (e.g. 'Sue Martin' with
# 'duane.martin...'), so the half-match is a common-surname coincidence, not a cleaning error.
raw_first = customers["customer_id"].map(raw.drop_duplicates("customer_id").set_index("customer_id")["first_name"])
assert raw_first[partial].str.replace(r"[^A-Za-z]", "", regex=True).str.lower().eq(
    customers.loc[partial, "first_name"].str.lower()).all()

mismatch = has_local & ~(first_ok & last_ok)
n_mismatch = flag(customers, mismatch, "email", "email name does not match customer name (email may belong to someone else)")
print(f"  flagged as email of another person: {n_mismatch:,}")

possible duplicates: same email 22 | same name + dob 2
emails in first.last format: 45,700
  first name matches email: 93.02%
  last name matches email:  93.02%
  only one of the two matches: 21


  flagged as email of another person: 3,200


## 13. Final table and validation

Every flag gets a `flag_type`, so the client sees at once what needs a decision:
- **needs review**: a person should check or decide (possible duplicate, suspicious value)
- **invalid or missing value**: the source value was impossible, a placeholder or blank; it was removed or left unconverted and cannot be recovered
- **auto-fixed (info)**: the value was repaired automatically and is listed for transparency

In [13]:
flag_table = pd.concat(flags, ignore_index=True)
# phones that could not be converted are invalid source values; everything else needs a person to decide
flag_table["flag_type"] = np.where(flag_table["reason"].str.contains("not converted"),
                                   "invalid or missing value", "needs review")
flag_table = flag_table[["customer_id", "flag_type", "column", "value", "reason"]]
dq = flag_table.groupby("customer_id")["reason"].agg("; ".join)
review_ids = set(flag_table.loc[flag_table["flag_type"].eq("needs review"), "customer_id"])

customers_clean = customers.assign(
    needs_review=customers["customer_id"].isin(review_ids),
    dq_flags=customers["customer_id"].map(dq).fillna(""),
)[[
    "customer_id", "first_name", "last_name", "email", "email_is_shared",
    "phone_e164", "phone_ext", "phone_raw", "gender", "dob", "signup_date",
    "address", "city", "state", "country", "state_country_conflict",
    "device_count", "source", "needs_review", "dq_flags",
]].reset_index(drop=True)

checks = {
    "one row per customer_id": customers_clean["customer_id"].is_unique,
    "all 48,200 distinct raw IDs kept": set(customers_clean["customer_id"]) == set(raw["customer_id"]),
    "every raw row accounted for in merge map": len(merge_map) == len(raw) and merge_map["action"].ne("").all(),
    "names contain letters, spaces, hyphens, apostrophes only":
        customers_clean[["first_name", "last_name"]].apply(lambda s: s.str.fullmatch(r"[A-Za-z '\-]+")).all().all(),
    "names in title case":
        customers_clean[["first_name", "last_name"]].apply(lambda s: s.eq(s.str.title())).all().all(),
    "no leading/trailing spaces in text columns":
        all(customers_clean[c].dropna().astype(str).eq(customers_clean[c].dropna().astype(str).str.strip()).all()
            for c in customers_clean.select_dtypes(exclude=["datetime", "bool", "number"]).columns),
    "phone_e164 always +1 followed by 10 digits":
        customers_clean["phone_e164"].dropna().str.fullmatch(r"\+1[2-9]\d{2}[2-9]\d{6}").all(),
    "devices table covers every customer": set(customer_devices["customer_id"]) == set(customers_clean["customer_id"]),
}
validation = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
assert validation["passed"].all(), validation
validation

,check,passed
0,one row per customer_id,True
1,"all 48,200 distinct raw IDs kept",True
2,every raw row accounted for in merge map,True
3,"names contain letters, spaces, hyphens, apostr...",True
4,names in title case,True
5,no leading/trailing spaces in text columns,True
6,phone_e164 always +1 followed by 10 digits,True
7,devices table covers every customer,True


In [14]:
customers_clean.head(10)

,customer_id,first_name,last_name,email,email_is_shared,phone_e164,phone_ext,phone_raw,gender,dob,signup_date,address,city,state,country,state_country_conflict,device_count,source,needs_review,dq_flags
0,0d244537-8164-4b84-bcf2-0e43766f3221,Maria,Day,maria.day697@hotmail.com,False,+14499771729,282,449.977.1729x282,M,1988-04-11,2016-12-05,107 Joseph Station,Padillaborough,Oregon,Micronesia,True,4,referral,False,
1,66c5800b-ad5c-4fac-86df-16472d82f02d,Kevin,Cantu,kevin.cantu968@gmail.com,False,+15013474824,64633,501.347.4824x64633,M,1955-09-19,2024-02-03,3577 Anita Knoll Apt. 107,Justinton,Delaware,Portugal,True,1,referral,False,
2,e7852eee-0a28-46dc-aa34-445a02530746,Ronald,Daniel,ronald.daniel138@yahoo.com,False,+12688226332,26814,+1-268-822-6332x26814,M,1971-11-02,2016-04-12,3440 Tammy Views Suite 698,New Autumnland,Mississippi,Uganda,True,1,web,False,
3,14f0e67d-e60e-4989-88cf-5be50d515a69,Katherine,Smith,katherine.smith357@yahoo.com,False,+18648941347,NaN,864-894-1347,F,1955-03-25,2019-06-21,98888 Morales Lakes,New Jose,Minnesota,Bahrain,True,1,web,False,
4,1ce2f061-345c-4121-a4ba-32963cf9a15d,Brandon,Glover,brandon.glover804@hotmail.com,False,+14897325436,NaN,489.732.5436,M,1968-06-25,2022-01-09,55037 Denise Crest Apt. 565,Port Amber,Oregon,Latvia,True,1,referral,False,
5,6ddc1772-f210-4111-87c9-c2ec73776c37,Toni,Li,toni.li092@gmail.com,False,+15223828559,NaN,(522)382-8559,F,1981-09-30,2023-11-25,1230 Padilla Parks Suite 647,North Melvintown,Kentucky,Uganda,True,1,app,False,
6,9a3ae2d2-97d7-43b4-843c-ebac2fc0131c,Barbara,Luna,barbara.luna614@yahoo.com,False,NaN,4636,001-063-045-0948x4636,F,1991-01-07,2025-11-01,768 Martha Plaza,Andersonberg,Maine,Guyana,True,1,referral,False,invalid US area code (starts with 0 or 1); not...
7,69814e99-477d-48dc-8d7b-2d1137f1cf37,Richard,Benson,richard.benson220@yahoo.com,False,+17552810372,NaN,755.281.0372,M,2002-08-02,2016-02-27,8812 Cheryl Radial Apt. 428,Huntview,Wyoming,Greenland,True,1,web,False,
8,d081d287-527e-4a77-bd62-50e28a110ddb,James,Ali,james.ali627@yahoo.com,False,NaN,NaN,1974876994,M,1968-02-11,2018-04-08,674 Lynch Spring Apt. 341,Martinstad,New Hampshire,Netherlands,True,1,app,False,invalid US area code (starts with 0 or 1); not...
9,5a298d80-e951-4565-98cb-5cb7ef301fc2,Jay,Jones,jay.jones624@yahoo.com,False,+18949336997,NaN,+1-894-933-6997,M,1963-08-22,2023-03-05,82891 Grant Park Suite 174,Brownfort,Wisconsin,Liechtenstein,True,1,web,False,


In [15]:
print(f"raw rows: {len(raw):,}")
print(merge_map["action"].value_counts().to_string())
print(f"golden records: {len(customers_clean):,}")
print(f"customers with at least one review flag: {customers_clean['dq_flags'].ne('').sum():,}")
print(f"customers that need review: {customers_clean['needs_review'].sum():,}")
flag_table.groupby(["flag_type", "reason"]).size().rename("rows")

raw rows: 50,000
action
kept: golden record                                        48200
dropped: exact duplicate of an earlier row                  1021
merged: same customer_id, identical after normalisation      779
golden records: 48,200
customers with at least one review flag: 20,560
customers that need review: 4,902


flag_type                 reason                                                                    
invalid or missing value  invalid US area code (starts with 0 or 1); not converted to E.164             9873
                          invalid US exchange code (digit 4 is 0 or 1); not converted to E.164          7815
needs review              customer younger than 13 at signup                                            1816
                          email name does not match customer name (email may belong to someone else)    3200
                          same name and date of birth on different customer_id (possible duplicate)        2
                          same personal email on different customer_id (possible duplicate)               22
Name: rows, dtype: int64

## 14. Save outputs

In [16]:
def write_xlsx(path, sheets):
    with pd.ExcelWriter(path, engine="xlsxwriter", date_format="yyyy-mm-dd", datetime_format="yyyy-mm-dd") as writer:
        for name, data in sheets.items():
            data.to_excel(writer, sheet_name=name, index=False)
            ws = writer.sheets[name]
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, len(data), len(data.columns) - 1)
            for i, col in enumerate(data.columns):
                sample = data[col].head(2000).astype(str).str.len().max() if len(data) else 0
                ws.set_column(i, i, min(max(sample, len(col)) + 2, 60))
    print("saved:", path.resolve())


fix_log_df = pd.DataFrame(fix_log)

write_xlsx(CLEAN_DIR / "customers_clean.xlsx", {
    "customers": customers_clean,
    "customer_devices": customer_devices,
    "flagged": flag_table.sort_values(["flag_type", "reason", "customer_id"]),
    "merge_map": merge_map,
})

CSV_DIR = CLEAN_DIR / "csv"
CSV_DIR.mkdir(exist_ok=True)
customers_clean.to_csv(CSV_DIR / "customers_clean.csv", index=False, encoding="utf-8", date_format="%Y-%m-%d")
customer_devices.to_csv(CSV_DIR / "customer_devices.csv", index=False, encoding="utf-8")
print("saved CSV:", ", ".join(sorted(p.name for p in CSV_DIR.glob("customer*.csv"))))
write_xlsx(OUT_DIR / "m2_fix_log.xlsx", {"fix_log": fix_log_df, "validation": validation})

saved: C:\Users\msibr\OneDrive\Upwork\customer360-data-cleaning\03_clean_data_and_report\clean_data\customers_clean.xlsx


saved CSV: customer_devices.csv, customers_clean.csv
saved: C:\Users\msibr\OneDrive\Upwork\customer360-data-cleaning\02_process\02_cleaning_customers-pandas\output\m2_fix_log.xlsx


## 15. Fix log

In [17]:
fix_log_df

,issue_id,column,issue,rows_in_m1,rows_fixed,rows_flagged,action_taken
0,ISS-017,(all),exact duplicate row,1021,1021,0,"Dropped, first occurrence kept"
1,ISS-008,first_name,leading/trailing spaces,3124,3124,0,Trimmed
2,ISS-007,last_name,leading/trailing spaces,3134,3134,0,Trimmed
3,ISS-009,first_name,noise characters (digits or symbols) in name,3104,3104,0,Removed digits and symbols
4,ISS-013,first_name,possible doubled-letter typo (e.g. 'KKeevvin'),1582,1576,0,"Rebuilt from letters 1, 3, 5 + rest"
5,ISS-006,first_name,inconsistent letter case,6272,4696,0,Converted to title case
6,ISS-010,last_name,noise characters (digits or symbols) in name,3086,3086,0,Removed digits and symbols
7,ISS-012,last_name,possible doubled-letter typo (e.g. 'KKeevvin'),1584,1581,0,"Rebuilt from letters 1, 3, 5 + rest"
8,ISS-005,last_name,inconsistent letter case,6280,4699,0,Converted to title case
9,ISS-016,email,blank value,1040,0,1018,Set to null (no reliable source to fill)
